In [33]:
from typing import Optional
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
import json
from reviews_api import get_product_rating
import base64
load_dotenv()

True

In [34]:
DB_path = os.path.join(os.getcwd(), "store.db")

print(DB_path)

d:\ksn-II\PRODUCTION\store.db


In [35]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0) 
vision_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [36]:


import sqlite3
@tool
def search_products(query:str, max_price:Optional[float]=None, is_organic:Optional[bool]=None) :
    """
    Search the product database by the keyword (matchd against name , description and category).
    operationally filter by maximum price and/or organic status.
    Return a Json array of matching products, each with :id ,name, category, price,description, is_organic.

    """

    conn = sqlite3.connect(DB_path)
    cursor = conn.cursor()

    sql="SELECT id,name,category,price ,description,is_organic FROM products Where 1=1"
    params : list=[]

    if query:
        sql += " AND name LIKE ?"
        params.append(f"%{query}%")

    if max_price is not None:
        sql += " AND price <= ?"
        params.append(max_price)

    if is_organic is not None:
        sql += " AND is_organic = ?"
        params.append(1 if is_organic else 0)

    cursor.execute(sql,params)
    rows=cursor.fetchall()
    conn.close()


    products=[
        {
            "id": row[0],
            "name": row[1],
            "category":row[2],
            "price": row[3],
            "description":row[4],
            "is_organic": bool(row[5]),
        }
        for row in rows
            

    ] 
    return json.dumps(products)


In [37]:
@tool
def get_rating(product_id:int)->str:
    """
    Get the average customer rating and total review count for a product by its ID. Returns a JSON object
    with: product_id,average_rating,review_count.
    """

    result= get_product_rating(product_id)
    return json.dumps(result)

In [42]:
@tool
def checkout(product_id: int) -> str:
    """
    Place an order for the given product ID. Saves the order to the database and returns a confirmation message
    with the order ID, product name, and price.
    """
    conn = sqlite3.connect(DB_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name, price FROM products WHERE id = ?", (product_id,))
    row = cursor.fetchone()

    if not row:
        conn.close()
        return f"Error: product with ID {product_id} not found."

    name, price = row
    cursor.execute(
        "INSERT INTO orders (product_id, product_name, price) VALUES (?, ?, ?)",
        (product_id, name, price),
    )
    order_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return f"Order #{order_id} confirmed: {name} — ₹{price}"

In [39]:
@tool 
def describe_product_image(image_path:str)->str:
    """
    Analyze a product image and return its key attributes as a json objects. use this when the user 
    uploads a photo or a product they are interested in. the returned attributes can be used directly 
    with search_products.
    """

    if image_path.startswith("http://") or image_path.startswith("https://"):
        import requests
        image_data = base64.b64encode(requests.get(image_path).content).decode()
        ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    else:
        with open(image_path,"rb") as f:
            image_data= base64.b64encode(f.read()).decode()
        ext=os.path.splitext(image_path)[1].lower().lstrip(".")
    mime= "image/jpeg" if ext in ("jpg","jpeg") else f"image/{ext}"

    message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "Look at this product image and extract its key attributes. "
                "Return ONLY a JSON object with these fields:\n"
                "- product_type: what kind of product it is (e.g. honey, olive oil)\n"
                "- search_query: a short keyword to search for it\n"
                "- is_organic: true if the label says organic, false if not, null if unclear\n"
                "- description: one sentence describing the product"
            ),
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime};base64,{image_data}"
            },
        },
    ]
)

    response = vision_llm.invoke([message])
    return response.content


In [40]:
agent = create_agent(
    tools=[search_products, get_rating, checkout,describe_product_image],
    model=llm,
    system_prompt=(
        "You are a helpful shopping assistant. Follow these rules strictly.\n\n"
        "IMAGE SEARCH — when the user provides an image path:\n"
        "1. Call describe_product_image with the path to identify the product.\n"
        "2. Use the returned search_query and is_organic to call search_products.\n"
        "3. Continue with the BrowING flow from step 2 onwards.\n\n"
        "BrowING — when the user describes what they want to buy:\n"
        "1. Call search_products to find matching items (apply any price/organic filters given).\n"
        "2. For each candidate, call get_rating to retrieve its average rating.\n"
        "3. Filter by the user's minimum rating if specified.\n"
        "4. Present qualifying products as a numbered list. For each item use this exact format "
        "   (plain text, no backticks, no code blocks, no bold, no italic):\n\n"
        "   #<number>. <name> (ID:<product_id>) — $<price> ★<rating> — <organic or non-organic>\n\n"
        "   Add a blank line between each product entry for readability. "
        "   Always include (ID:X) so you can reference it later.\n"
        "5. If only one product qualifies, still show it in the list and ask: "
        "   'Would you like to order it? Just say yes or give me the number.'\n"
        "6. Do NOT call checkout at this stage.\n\n"
        "ORDERING — when the user confirms they want to buy (e.g. 'yes', 'sure', 'go ahead', "
        "'order number 2', 'the first one', 'get me #3'):\n"
        "1. Look at your previous message to find the (ID:X) for the chosen product "
        "   (if only one was listed and the user says 'yes', use that product's ID).\n"
        "2. Call checkout with that product_id (the number from (ID:X)).\n"
        "3. Confirm the order to the user in plain text.\n\n"
        "Never place an order unless the user explicitly confirms. "
        "Never guess a product_id — always take it from the (ID:X) in your own previous message."
    ),
)

In [ ]:
result = search_products.invoke({
    "query": "honey",
    "max_price": None,
    "organic_only": False
})

print(result)

[{"id": 1, "name": "Organic Raw Honey", "category": "honey", "price": 14.99, "description": "Pure organic raw honey, unfiltered and cold-pressed", "is_organic": true}, {"id": 2, "name": "Wildflower Honey", "category": "honey", "price": 12.99, "description": "Natural wildflower honey from local beekeepers", "is_organic": false}, {"id": 3, "name": "Organic Manuka Honey", "category": "honey", "price": 29.99, "description": "Premium organic Manuka honey from New Zealand", "is_organic": true}, {"id": 4, "name": "Clover Honey", "category": "honey", "price": 8.99, "description": "Classic clover honey, smooth and sweet", "is_organic": false}, {"id": 5, "name": "Organic Buckwheat Honey", "category": "honey", "price": 18.99, "description": "Dark and robust organic buckwheat honey, antioxidant-rich", "is_organic": true}, {"id": 6, "name": "Orange Blossom Honey", "category": "honey", "price": 15.99, "description": "Light and floral orange blossom honey", "is_organic": false}, {"id": 7, "name": "Or

In [ ]:
print(get_rating.invoke({"product_id": 1}))

{"product_id": 1, "average_rating": 4.62, "review_count": 4}


In [ ]:
print(type(describe_product_image))

<class 'function'>
